In [90]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [91]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [92]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task, format and solution criteria fields that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python/json/regex",
        "solution_criteria": "key criteria to evaluate the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on concise solution_criteria that can be used to evaluate the correctness of the solution.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages,stop_sequences=["```"])
    return json.loads(response)

In [93]:
datasets = generate_dataset()
# store the dataset in a json file
with open("dataset.json", "w") as f:
    json.dump(datasets, f, indent=4)

In [94]:
def run_prompt(test_case):
    """Merges prompt with test case and runs the prompt, returning the results."""
    prompt = f"""
        Please solve the following task:
        {test_case['task']}

        * respond only with either a Python function, a JSON object, or a regular expression depending on the format specified in the test case.
        * do not include any explanations, only respond with the required format.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    response = chat(messages,stop_sequences=["```"])
    return response

In [95]:
def grade_by_model(test_case, response):
    """Grades the response using the model."""
    prompt = f"""
        You are an expert AWS grader. Your task is to evaluate the following AI-generated response to an AWS-related task. 

        Original Task:
        <task>
        {test_case['task']}
        </task>

        Solution Criteria:
        <criteria>
        {test_case['solution_criteria']}
        </criteria>

        Solution to Evaluate:
        <solution>
        {response}
        </solution>

        Output Format:
        Provide your evaluation based on the solution_criteria provided in structured JSON object with the following field, in this specific order:
        - strength: An array of 1-3 key strengths of the response based on the solution criteria.
        - weakness: An array of 1-3 key weaknesses of the response based on the solution criteria.
        - reasoning: A concise explanation of the assessment.
        - score: A number between 1-10, with 1 being worst and 10 being best.
        Please grade the following response to the task on a scale of 1 to 10, with 10 being the best. 
        Example response shape:
        {{
            "strength": string[],
            "weakness": string[],
            "reasoning": string,
            "score": number
        }}
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [96]:
import re
import ast

def validate_json(response):
    """Validates if the response is a valid JSON object."""
    try:
        json.loads(response)
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_regex(response):
    """Validates if the response is a valid regular expression."""
    try:
        re.compile(response)
        return 10
    except re.error:
        return 0
    
def validate_python(response):
    """Validates if the response is a valid Python function."""
    try:
        ast.parse(response)
        return 10
    except SyntaxError:
        return 0
    
def grade_by_syntax(test_case, response):
    """Grades the response based on the expected format."""
    if test_case['format'] == 'json':
        return validate_json(response)
    elif test_case['format'] == 'regex':
        return validate_regex(response)
    elif test_case['format'] == 'python':
        return validate_python(response)
    else:
        return 0

In [97]:
def run_task(test_case):
    """Calls run_prompt and returns the results."""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade['score']
    reasoning = model_grade['reasoning']

    syntax_score = grade_by_syntax(test_case, output)

    score = (model_score + syntax_score) / 2
    
    return  dict(
        output=output, test_case=test_case, score=score, reasoning=reasoning
    )
    

In [98]:
def run_eval(dataset):
    """Run the evaluation on the dataset and return the results."""
    results = []
    for test_case in dataset:
        result = run_task(test_case)
        results.append(result)
    return results

In [99]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


In [100]:
# print the average score
from statistics import mean
print(json.dumps(results, indent=4))
average_score = mean([result['score'] for result in results])
print(f"Average Score: {average_score}")


[
    {
        "output": "\nimport re\n\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI string.\n    \n    Args:\n        s3_uri: S3 URI string in format 's3://bucket-name/path/to/object'\n    \n    Returns:\n        The bucket name\n    \"\"\"\n    match = re.match(r's3://([^/]+)', s3_uri)\n    if match:\n        return match.group(1)\n    return None\n",
        "test_case": {
            "task": "Parse an AWS S3 bucket name from an S3 URI string (e.g., 's3://my-bucket-name/path/to/object')",
            "format": "regex",
            "solution_criteria": "Regex must correctly extract bucket name from valid S3 URIs and handle URIs with or without object paths"
        },
        "score": 8.5,
        "reasoning": "The solution correctly implements the core requirement of parsing S3 bucket names from S3 URIs using a well-crafted regex pattern that handles both formats with and without object paths. The regex logic is sound an